## Dynamic Adaptive Task Decomposition with MAF and Foundry Agents

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.10

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

credential = AzureCliCredential()

### Creating the Foundry Chat Client

In [ ]:
import os
import json
import asyncio

from typing import Literal
from pydantic import BaseModel, Field

from agent_framework import Agent

from agent_framework.foundry import (
    FoundryChatClient
)

client = FoundryChatClient(
    project_endpoint = foundry_project_endpoint,
    model = model_deployment_name,
    credential = credential
)

await client.configure_azure_monitor(
    enable_sensitive_data=True
)

### Create the Web Search Tool

In [ ]:
# create the web search tool for the researcher agent
web_search_tool = (
    FoundryChatClient.get_web_search_tool(
        search_context_size="medium"
    )
)

### Create the Web Research Agent

In [ ]:
research_agent = Agent(
    client=client,

    name="ResearchAgent",

    description=(
        "Research specialist that finds current "
        "information using web search."
    ),

    instructions="""
    You are a research specialist.

    Complete ONLY the research task assigned to you.

    Use Web Search to find current and reliable
    information.

    Prefer authoritative and primary sources.

    Collect useful:

    - facts
    - statistics
    - dates
    - numerical values
    - trends
    - recent developments

    Preserve numerical information carefully.

    Always preserve:

    - units
    - currencies
    - percentages
    - dates
    - time periods

    Your output will be consumed by other agents,
    so make it factual, structured and concise.
    """,

    tools=[
        web_search_tool
    ]
)

### Create the Data Analysis Agent

In [ ]:
analysis_agent = Agent(
    client=client,

    name="AnalysisAgent",

    description=(
        "Analyzes research findings and identifies "
        "comparisons, patterns and important insights."
    ),

    instructions="""
    You are an analytical specialist.

    You receive research produced by other agents.

    Analyze ONLY the information supplied to you.

    You do NOT have web access.

    Do not invent missing facts.

    Your responsibilities include:

    - comparing findings
    - identifying patterns
    - calculating simple differences when possible
    - identifying important trends
    - identifying similarities and differences
    - explaining what the evidence indicates

    Preserve important numerical information.

    Return structured analytical findings that can
    be consumed by the final synthesis agent.
    """
)

### Create the Synthesis Agent

In [ ]:
synthesis_agent = Agent(
    client=client,

    name="SynthesisAgent",

    description=(
        "Combines research and analysis into "
        "a final answer."
    ),

    instructions="""
    You are the final synthesis specialist.

    You receive outputs produced by specialist agents.

    Combine those outputs into a clear and polished
    response to the user's original request.

    Use ONLY the supplied specialist results.

    Do not perform new research.

    Do not invent missing facts or statistics.

    Resolve repetition and organize the information
    logically.

    Structure the final answer as:

    # Executive Summary

    # Key Findings

    # Analysis

    # Conclusion
    """
)

### Create the Agent Registry

In [ ]:
AGENT_REGISTRY = {

    "research_agent": {

        "description":
            "Finds current external information "
            "using web search.",

        "capabilities": [
            "web search",
            "current information",
            "company research",
            "technology research",
            "market research",
            "finding statistics",
            "finding recent developments"
        ]
    },


    "analysis_agent": {

        "description":
            "Analyzes supplied research findings.",

        "capabilities": [
            "comparison",
            "reasoning",
            "trend analysis",
            "simple calculations",
            "identifying patterns",
            "interpreting evidence"
        ]
    },


    "synthesis_agent": {

        "description":
            "Combines specialist outputs into "
            "the final response.",

        "capabilities": [
            "synthesis",
            "report writing",
            "summarization",
            "combining findings"
        ]
    }
}

### Define the Structured Output Schema

In [ ]:
class SubTask(BaseModel):

    task_id: str = Field(
        description="Unique task ID such as T1"
    )

    description: str = Field(
        description="What this subtask must accomplish"
    )

    assigned_agent: Literal[
        "research_agent",
        "analysis_agent",
        "synthesis_agent"
    ]

    depends_on: list[str] = Field(
        default_factory=list,
        description="Task IDs that must finish first"
    )

    expected_output: str = Field(
        description="Expected result from this task"
    )


class ExecutionPlan(BaseModel):

    complexity: Literal[
        "simple",
        "moderate",
        "complex"
    ]

    reasoning: str

    subtasks: list[SubTask]

### Create the Planner Agent

In [ ]:
planner_agent = Agent(
    client=client,

    name="PlannerAgent",

    description=(
        "Meta-agent that dynamically decomposes "
        "requests into specialist subtasks."
    ),

    instructions=f"""
    You are a dynamic task decomposition planner.

    You DO NOT execute the user's request.

    Your only responsibility is to create an
    execution plan.

    AVAILABLE AGENTS
    =================

    {json.dumps(
        AGENT_REGISTRY,
        indent=2
    )}


    PLANNING RULES
    ==============

    1. Create only necessary subtasks.

    2. Use research_agent whenever current
       external information must be discovered.

    3. You may create MULTIPLE independent
       research tasks.

    4. Independent research tasks should have
       no dependencies so they can run in parallel.

    5. Use analysis_agent when multiple research
       results need comparison or interpretation.

    6. Use synthesis_agent to combine the completed
       work into the final answer.

    7. The final task should normally be assigned
       to synthesis_agent.

    8. Specify dependencies correctly.

    Example:

       T1:
       Research Microsoft
       depends_on = []

       T2:
       Research Google
       depends_on = []

       T3:
       Compare Microsoft and Google
       depends_on = ["T1", "T2"]

       T4:
       Produce final answer
       depends_on = ["T1", "T2", "T3"]

    9. Do not create circular dependencies.

    10. Every dependency must refer to another
        task in the plan.

    11. Keep simple requests simple.

    12. Use deeper decomposition only when the
        request actually requires it.
    """
)

### Generate the Dynamic Plan

In [ ]:
async def create_plan(
    query
):

    response = await planner_agent.run(
        query,

        options={
            "response_format":
                ExecutionPlan
        }
    )

    return response.value

In [ ]:
def print_plan(
    plan
):

    print(
        "\n======================================"
    )

    print(
        "       DYNAMIC EXECUTION PLAN"
    )

    print(
        "======================================"
    )


    print(
        f"\nComplexity: "
        f"{plan.complexity.upper()}"
    )


    print(
        f"\nPlanner reasoning:\n"
        f"{plan.reasoning}"
    )


    for task in plan.subtasks:

        print(
            "\n--------------------------------------"
        )

        print(
            f"{task.task_id}"
        )

        print(
            f"Agent: "
            f"{task.assigned_agent}"
        )

        print(
            f"Task: "
            f"{task.description}"
        )

        print(
            f"Depends on: "
            f"{task.depends_on or 'None'}"
        )

        print(
            f"Expected output: "
            f"{task.expected_output}"
        )

### Validate the Plan for Dependencies and Conflicts

In [ ]:
AGENTS = {

    "research_agent":
        research_agent,

    "analysis_agent":
        analysis_agent,

    "synthesis_agent":
        synthesis_agent
}

In [ ]:
def validate_plan(
    plan
):

    task_ids = [
        task.task_id
        for task in plan.subtasks
    ]


    # Duplicate IDs
    if len(task_ids) != len(set(task_ids)):

        raise ValueError(
            "Plan contains duplicate task IDs."
        )


    valid_ids = set(
        task_ids
    )


    # Check dependencies
    for task in plan.subtasks:

        for dependency in task.depends_on:

            if dependency not in valid_ids:

                raise ValueError(
                    f"{task.task_id} depends on "
                    f"unknown task {dependency}."
                )


            if dependency == task.task_id:

                raise ValueError(
                    f"{task.task_id} cannot "
                    f"depend on itself."
                )


    print(
        "✓ Execution plan validated"
    )

### Define the Execution of a Subtask

In [ ]:
async def execute_subtask(
    task,
    original_query,
    completed_tasks
):

    agent = AGENTS[
        task.assigned_agent
    ]


    # ------------------------------------------
    # Collect dependency outputs
    # ------------------------------------------

    dependency_context = {

        dependency_id:
            completed_tasks[
                dependency_id
            ]

        for dependency_id
        in task.depends_on
    }


    # ------------------------------------------
    # Build specialist prompt
    # ------------------------------------------

    prompt = f"""
    ORIGINAL USER REQUEST
    =====================

    {original_query}


    YOUR ASSIGNED SUBTASK
    =====================

    Task ID:
    {task.task_id}

    Task:
    {task.description}


    EXPECTED OUTPUT
    ===============

    {task.expected_output}


    RESULTS FROM DEPENDENCIES
    =========================

    {
        json.dumps(
            dependency_context,
            indent=2
        )
        if dependency_context
        else "No dependency results."
    }


    INSTRUCTIONS
    ============

    Complete ONLY your assigned subtask.

    Do not attempt to perform tasks assigned
    to other agents.

    Use the dependency results when relevant.

    Do not invent missing information.
    """


    print(
        f"\n▶ Starting "
        f"{task.task_id}"
    )

    print(
        f"  Agent: "
        f"{task.assigned_agent}"
    )


    # ------------------------------------------
    # Execute specialist
    # ------------------------------------------

    response = await agent.run(
        prompt
    )


    print(
        f"✓ Completed "
        f"{task.task_id}"
    )


    return response.text

### Create the Dependency Aware Executor

In [ ]:
async def execute_plan(
    plan,
    original_query
):

    completed_tasks = {}


    remaining_tasks = {

        task.task_id:
            task

        for task in plan.subtasks
    }


    execution_round = 1


    while remaining_tasks:


        # --------------------------------------
        # Find executable tasks
        # --------------------------------------

        ready_tasks = [

            task

            for task
            in remaining_tasks.values()

            if all(

                dependency
                in completed_tasks

                for dependency
                in task.depends_on
            )
        ]


        if not ready_tasks:

            raise RuntimeError(
                "No executable tasks remain. "
                "The plan may contain "
                "circular dependencies."
            )


        print(
            "\n======================================"
        )

        print(
            f"EXECUTION ROUND "
            f"{execution_round}"
        )

        print(
            "======================================"
        )


        print(
            "Ready tasks:",
            [
                task.task_id
                for task in ready_tasks
            ]
        )


        # --------------------------------------
        # Run independent tasks concurrently
        # --------------------------------------

        results = await asyncio.gather(

            *[

                execute_subtask(
                    task,
                    original_query,
                    completed_tasks
                )

                for task
                in ready_tasks
            ]
        )


        # --------------------------------------
        # Store results
        # --------------------------------------

        for task, result in zip(
            ready_tasks,
            results
        ):

            completed_tasks[
                task.task_id
            ] = result


            del remaining_tasks[
                task.task_id
            ]


        execution_round += 1


    return completed_tasks

### Define the Complete Runner Function

In [ ]:
async def run_dynamic_decomposition(
    query
):

    # ==========================================
    # 1. PLAN
    # ==========================================

    print(
        "\n🧠 Creating dynamic execution plan..."
    )


    plan = await create_plan(
        query
    )


    # ==========================================
    # 2. INSPECT PLAN
    # ==========================================

    print_plan(
        plan
    )


    validate_plan(
        plan
    )


    # ==========================================
    # 3. EXECUTE
    # ==========================================

    print(
        "\n🚀 Executing plan..."
    )


    results = await execute_plan(
        plan,
        query
    )


    # ==========================================
    # 4. FIND FINAL SYNTHESIS TASK
    # ==========================================

    synthesis_tasks = [

        task

        for task in plan.subtasks

        if task.assigned_agent
        == "synthesis_agent"
    ]


    if not synthesis_tasks:

        raise RuntimeError(
            "Planner did not create "
            "a synthesis task."
        )


    final_task = (
        synthesis_tasks[-1]
    )


    final_answer = (
        results[
            final_task.task_id
        ]
    )


    # ==========================================
    # 5. RETURN
    # ==========================================

    return {

        "plan":
            plan,

        "results":
            results,

        "final_answer":
            final_answer
    }

### Test with a Request

In [ ]:
query = """
Research the recent generative AI activities of
Microsoft, Google and Amazon.

Focus on developments during the last two years.

For each company:

- identify major generative AI products
- identify important platform developments
- find useful quantitative indicators
- identify major strategic developments

Then compare the three companies and identify
the most important trends.

Finally, produce a concise comparative report.
"""

In [ ]:
execution = await run_dynamic_decomposition(
    query
)

In [ ]:
print(
    "\n======================================"
)

print(
    "             FINAL ANSWER"
)

print(
    "======================================\n"
)


print(
    execution[
        "final_answer"
    ]
)